# Notebook 05: Isolated Model Training - Hospital A (Cardiology Focus)

## Objective
Train a local model exclusively on **Hospital A** dataset (which has Cardiology/Heart Disease bias).
Evaluate how an isolated hospital model performs locally vs. globally on other hospital nodes.



In [1]:
import os
import sys
# Ensure project root is in sys.path for backend and scripts imports
root_path = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if root_path not in sys.path:
    sys.path.insert(0, root_path)
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from backend.ml.model import DiabetesRiskModel

# Load datasets
hosp_a = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_A.csv"))
hosp_b = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_B.csv"))
hosp_c = pd.read_csv(os.path.join(root_path, "data", "hospitals", "hospital_C.csv"))

feature_cols = ['age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'bmi', 'glucose', 'hba1c', 'cholesterol', 'creatinine']
target_col = 'diabetes'

scaler = StandardScaler()
scaler.fit(pd.concat([hosp_a, hosp_b, hosp_c])[feature_cols])

X_a = scaler.transform(hosp_a[feature_cols].values)
y_a = hosp_a[target_col].values

# Split Hospital A locally
split_idx = int(len(X_a) * 0.8)
X_a_train, X_a_test = X_a[:split_idx], X_a[split_idx:]
y_a_train, y_a_test = y_a[:split_idx], y_a[split_idx:]

print(f"Hospital A Train: {len(X_a_train)}, Test: {len(X_a_test)}")


Hospital A Train: 411, Test: 103


C:\Users\bahad\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 1. Train Local Model on Hospital A


In [2]:
train_loader = DataLoader(TensorDataset(torch.tensor(X_a_train, dtype=torch.float32), torch.tensor(y_a_train, dtype=torch.float32)), batch_size=32, shuffle=True)

model_a = DiabetesRiskModel(input_dim=len(feature_cols))
optimizer = optim.Adam(model_a.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

model_a.train()
for epoch in range(25):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model_a(bx).squeeze(), by)
        loss.backward()
        optimizer.step()

print("Hospital A Local Training Complete.")


Hospital A Local Training Complete.


## 2. Evaluate Local Model A across Node Datasets


In [3]:
def eval_model_on_data(model, X_vals, y_vals):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_vals, dtype=torch.float32)).squeeze()
        probs = torch.sigmoid(logits).numpy()
        preds = (probs >= 0.5).astype(float)
    return {
        'acc': accuracy_score(y_vals, preds),
        'f1': f1_score(y_vals, preds, zero_division=0),
        'auc': roc_auc_score(y_vals, probs)
    }

# Hospital B and C test sets
X_b = scaler.transform(hosp_b[feature_cols].values)
y_b = hosp_b[target_col].values
X_c = scaler.transform(hosp_c[feature_cols].values)
y_c = hosp_c[target_col].values

res_a_on_a = eval_model_on_data(model_a, X_a_test, y_a_test)
res_a_on_b = eval_model_on_data(model_a, X_b, y_b)
res_a_on_c = eval_model_on_data(model_a, X_c, y_c)

print("Hospital A Model Performance:")
print(f"Tested on Hospital A (Local):  Acc = {res_a_on_a['acc']:.4f}, F1 = {res_a_on_a['f1']:.4f}")
print(f"Tested on Hospital B (Remote): Acc = {res_a_on_b['acc']:.4f}, F1 = {res_a_on_b['f1']:.4f}")
print(f"Tested on Hospital C (Remote): Acc = {res_a_on_c['acc']:.4f}, F1 = {res_a_on_c['f1']:.4f}")


Hospital A Model Performance:
Tested on Hospital A (Local):  Acc = 0.9126, F1 = 0.9280
Tested on Hospital B (Remote): Acc = 0.9208, F1 = 0.9447
Tested on Hospital C (Remote): Acc = 0.8985, F1 = 0.8188


C:\Users\bahad\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\bahad\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
